### CS/ECE/ISyE 524 &mdash; Introduction to Optimization &mdash; Spring 2026 ###

# Project title goes here #

#### Student 1 (email address), Student 2 (email address), Student 3 (email address)

### Table of Contents

1. [Introduction](#1.-Introduction)
1. [Mathematical Model](#2.-Mathematical-model)
1. [Implementation and Solution](#3.-Implementation)
1. [Discussion of Results](#4.-Results-discussion)
1. [Optional Subsection](#4.A.-Feel-free-to-add-subsections)
1. [Conclusion](#5.-Conclusion)

## 1. Introduction ##

The first few sentences should give a quick overview of the entire project. Then, elaborate with a description of the problem that will be solved, a brief history (with [citations](https://en.wikipedia.org/wiki/Citation)) of how the problem came about, why it's important/interesting, and any other interesting facts you'd like to talk about. You should address and explain where the problem data is coming from (research? the internet? synthetically generated?) Also give an outline of the rest of the report.

This section **should be accessible to a general audience** (don't assume your reader has taken the class!). Feel free to include images if you think it'll be helpful:

![fixit flowchart][flow]

For more help on using Markdown, see [this reference](https://github.com/adam-p/markdown-here/wiki/Markdown-Cheatsheet).

[flow]: https://s-media-cache-ak0.pinimg.com/736x/f5/75/c5/f575c53b93724808c6f0211890a54900.jpg

## 2. Mathematical model ##

A discussion of the modeling assumptions made in the problem (e.g. is it from physics? economics? something else?). Explain the decision variables, the constraints, and the objective function. Finally, show the optimization problem written in standard form. If converting the problem to standard form would create a difficult-to-read model, it is okay to leave it in its simplest original form. Equations should be formatted in $\\LaTeX$ within the IJulia notebook. For this section you may **assume the reader is familiar with the material covered in class**.

Here is an example of an equation:

$$\begin{bmatrix}
      1 & 2 \\
       3 & 4
    \end{bmatrix}
    \begin{bmatrix} x \\ y \end{bmatrix} =
    \begin{bmatrix} 5 \\ 6 \end{bmatrix}$$

And here is an example of an optimization problem in easy-to-read form:
$$\begin{aligned}
  \underset{x \in \mathbb{R^n}}{\text{maximize}}\qquad& f_0(x) \\
    \text{subject to:}\qquad& f_i(x) \le 0 && i=1,\dots,m\\
    & h_j(x) = 0 && j=1,\dots,r
    \end{aligned}$$


## 3. Implementation \& Solutions ##

Here, you should code up your model in Julia + JuMP and solve it. Your code should be clean, easy to read, well annotated and commented, and it should compile! You are not allowed to use other programming languages or DCP packages such as `convex.jl`. I suggest having multiple code blocks separated by text blocks that explain the various parts of your solution. You may want to solve several versions of your problem with different models/assumptions. Suppress any code output that you don't explain in the body of the report.

It's fine to call external packages such as `Gurobi`, but try to minimize the use of exotic libraries.

In [1]:
using Pkg
# Pkg.add(["JuMP", "HiGHS", "DataFrames", "CSV"])

using JuMP, HiGHS, DataFrames, CSV

"""
Function to run the diet optimization model.
Integrates nutritional constraints, cost, and carbon footprint.
"""
function run_optimized_diet()
    # 1. Load Datasets
    # food_df: Nutritional data (~300 items)
    # ghg_df: Carbon footprint data (~40 categories)
    food_df = CSV.read("food.csv", DataFrame)
    ghg_df = CSV.read("ghg-per-kg-poore.csv", DataFrame)

    # 2. Define the Mapping Dictionary
    # Keys: Food.csv Categories or Keywords
    # Values: GHG.csv Entities
    mapping = Dict(
        "Milk" => "Milk",
        "Beef" => "Beef (beef herd)",
        "Pork" => "Pig Meat",
        "Poultry" => "Poultry Meat",
        "Lamb" => "Lamb & Mutton",
        "Cheese" => "Cheese",
        "Eggs" => "Eggs",
        "Fish" => "Fish (farmed)",
        "Apples" => "Apples",
        "Bananas" => "Bananas",
        "Vegetables" => "Other Vegetables",
        "Fruits" => "Other Fruit",
        "Nuts" => "Nuts",
        "Grains" => "Wheat & Rye"
    )

    # Convert GHG emissions to a lookup dictionary for efficiency
    # Unit: kg CO2e per kg of food
    ghg_lookup = Dict(row.Entity => row."Greenhouse gas emissions per kilogram" for row in eachrow(ghg_df))

    # 3. Data Preprocessing & Alignment
    # We create a new column 'GHG_val' in the food dataframe based on the mapping
    food_df.GHG_per_100g = missings(Float64, nrow(food_df))
    
    for i in 1:nrow(food_df)
        cat = food_df[i, :Category]
        if haskey(mapping, cat)
            entity = mapping[cat]
            if haskey(ghg_lookup, entity)
                # GHG data is per kg, we convert to per 100g (divide by 10)
                food_df[i, :GHG_per_100g] = ghg_lookup[entity] / 10.0
            end
        end
    end

    # Filter out foods that couldn't be mapped to a carbon value
    optimized_pool = filter(row -> !ismissing(row.GHG_per_100g), food_df)
    n = nrow(optimized_pool)

    # 4. Initialize Optimization Model
    # Using HiGHS as the Linear Programming solver
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    # 5. Decision Variables
    # x[i]: Amount of food 'i' to consume daily (unit: 100g)
    @variable(model, x[1:n] >= 0)

    # 6. Objective Function (Minimize Carbon Footprint)
    # Total Emissions = Sum(Servings * Emission_per_Serving)
    @objective(model, Min, sum(x[i] * optimized_pool[i, :GHG_per_100g] for i in 1:n))

    # 7. Nutritional Constraints (Daily Requirements)
    # Minimum Protein Requirement: 50g
    @constraint(model, sum(x[i] * optimized_pool[i, :"Data.Protein"] for i in 1:n) >= 50.0)

    # Maximum Total Fat Limit: 65g
    @constraint(model, sum(x[i] * optimized_pool[i, :"Data.Fat.Total Lipid"] for i in 1:n) <= 65.0)
    
    # Sodium Limit (Hypertension Variant): Max 2300mg
    @constraint(model, sum(x[i] * optimized_pool[i, :"Data.Major Minerals.Sodium"] for i in 1:n) <= 2300.0)

    # Diversity Constraint: Limit each food to max 5 units (500g) to ensure a varied diet
    for i in 1:n
        @constraint(model, x[i] <= 5.0)
    end

    # 8. Execute Solver
    optimize!(model)

    # 9. Output Results
    if termination_status(model) == MOI.OPTIMAL
        println("Optimization Successful!")
        println("Total Daily Carbon Footprint: ", round(objective_value(model), digits=3), " kg CO2e")
        
        println("\n--- Recommended Daily Meal Plan ---")
        for i in 1:n
            servings = value(x[i])
            if servings > 0.1 # Show only items with more than 10g consumption
                desc = optimized_pool[i, :Description]
                grams = round(servings * 100, digits=1)
                println("- $desc: $grams g")
            end
        end
    else
        println("The model could not find an optimal solution. Please check the constraints.")
    end
end

# Run the model
run_optimized_diet()

Optimization Successful!
Total Daily Carbon Footprint: 0.282 kg CO2e

--- Recommended Daily Meal Plan ---
- Milk, dry, not reconstituted, fat free (skim): 73.2 g
- Nuts, NFS: 120.2 g


## 4. Discussion of Results ##

Here, you display and discuss the results. Show figures, tables, plots, images, trade-off curves, or whatever else you can think of to best illustrate your results. The discussion should explain what the results mean, and how to interpret them. You should also explain the limitations of your approach/model and how sensitive your results are to the assumptions you made.

 Use plots (see `Plots` examples from class), or you can display results in a table like this:

| Tables        | Are          | Cool  |
| ------------- |:-------------| -----:|
| col 3 is      |right-aligned |\$1600 |
|  colons       | align columns|  \$12 |
| zebra stripes |    are neat  |   \$1 |

### 4.A. Feel free to add subsections

#### 4.A.a. or subsubsections

## 5. Conclusion ##

Summarize your findings and your results, and talk about at least one possible future direction; something that might be interesting to pursue as a follow-up to your project.